1. Nucleos puntuales: Representan toda la masa de cada galaxia, es decir las interacciones gravitacional se dan con y entre estos nucleos.

2. Estrellas como particulas de prueba (m=0), no interactuan entre ellas

In [18]:
import numpy as np
import matplotlib.pyplot as plt
from numba import jit

In [19]:
class N_System:
    def __init__(self, particles: list[dict[str, any]]):
        self.n = len(particles)
        self.positions = np.array([p['position'] for p in particles], dtype=np.float64)
        self.velocities = np.array([p['velocity'] for p in particles], dtype=np.float64)
        self.masses = np.array([p['mass'] for p in particles], dtype=np.float64)
        self.mass = self.masses.sum()

        v_prom = np.mean(np.linalg.norm(self.velocities, axis=1))
        self.t_relax = self.n / np.log(self.n) * (v_prom**3 / self.mass)

        self.mass_indices = np.where(self.masses > 0)[0]
    
    def evolucionar(self, dt: float, T: float) -> np.ndarray[float,3]:
        #Metodo de Leapfrog
        def aceleracion(pos:np.ndarray, masses:np.ndarray[float],m_indices:np.ndarray[int],n_particles:int) -> np.ndarray[float,3]:
            acc = np.zeros_like((n_particles, 3), dtype=np.float64)
            for i in range(self.n):
                for j in m_indices:
                    if i == j:
                        continue
                    r_vec = pos[i] - pos[j]
                    r_mag = np.linalg.norm(r_vec)
                    acc -= masses[j] * r_vec / r_mag**3
            return acc

        num_steps = int(T / dt)
        trajectories = np.zeros((self.n, num_steps, 3))
        
        accelerations = aceleracion(self.positions, self.masses, self.mass_indices, self.n)
        self.velocities += 0.5 * accelerations * dt

        for step in range(1,num_steps):
            self.positions += self.velocities * dt
            trajectories[:,step,:] = self.positions
            accelerations = aceleracion(self.positions, self.masses, self.mass_indices, self.n)
            self.velocities += accelerations * dt   
        return trajectories        

            


    

In [20]:
Primer_sistema = N_System([
    {'position': [0, 0, 0], 'velocity': [0, 0, 0], 'mass': 1},                
    {'position': [1, 0, 0], 'velocity': [0, 1, 0], 'mass': 1},                
    {'position': [0, 1, 0], 'velocity': [-1, 0, 0], 'mass': 1},                
    {'position': [1, 1, 0], 'velocity': [0, -1, 0], 'mass': 1}])

In [22]:
Primer_sistema.evolucionar(0.01, 10)

ValueError: operands could not be broadcast together with shapes (2,) (3,) (2,) 